# 3 · Playback: score the test week
Scores every station every 15 minutes of the held-out test week with the `champion` models, next to what actually happened in the following 15 minutes. The map app replays this table, so you can watch predictions and outcomes side by side.

**Needs:** notebook 2 in production. **Writes:** `team1_playback_15`.

In [ ]:
# Settings and shared code
import mlflow
import mlflow.sklearn
import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F

import mvp_config as cfg
import mvp_lib as lib

spark.conf.set("spark.sql.session.timeZone", "UTC")
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
PUBLISH = dbutils.widgets.get("run_mode") == "production"
mlflow.set_registry_uri("databricks-uc")

info = spark.table(cfg.MODEL_INFO).toPandas().set_index("target")
models = {t: mlflow.sklearn.load_model(f"models:/{cfg.MODEL_NAMES[t]}@{cfg.MODEL_ALIAS}") for t in ("bike", "dock")}
thresholds = {t: float(info.loc[t, "threshold"]) for t in ("bike", "dock")}
versions = {t: str(info.loc[t, "version"]) for t in ("bike", "dock")}
test_start = pd.Timestamp(info.loc["bike", "test_start"])
for t in ("bike", "dock"):
    print(f"{t}: {info.loc[t, 'model_type']} v{versions[t]} · warning threshold {thresholds[t]:.3f}")
print("test week starts", test_start)

In [ ]:
# One poll per station per 15-minute slot (the latest in the slot), with outcomes.
step = cfg.PLAYBACK_STEP_MIN * 60
slot = F.timestamp_seconds(F.floor(F.unix_seconds("fetched_at") / step) * step)
latest = Window.partitionBy("station_id", "prediction_time").orderBy(F.desc("fetched_at"))

columns = sorted(set(lib.all_input_columns()) | {"station_id", "fetched_at", "capacity",
                                                  "bike_stockout", "dock_stockout"})
polls = (spark.table(cfg.PREDICTORS).select(*columns)
         .filter(F.col("fetched_at") >= F.lit(test_start.to_pydatetime()))
         .withColumn("prediction_time", slot)
         .withColumn("_rank", F.row_number().over(latest)).filter("_rank = 1").drop("_rank")
         .join(spark.table(cfg.TARGETS).select("station_id", "fetched_at", "bike_target", "dock_target"),
               ["station_id", "fetched_at"], "left"))

days = [r[0] for r in polls.select(F.to_date("prediction_time").alias("d")).distinct().orderBy("d").collect()]
print(f"{len(days)} days to score")

In [ ]:
# Score one day at a time so each pandas frame stays small.
OUT = ["station_id", "prediction_time", "fetched_at", "num_bikes_available", "num_docks_available",
       "capacity", "bike_state", "dock_state", "p_bike", "p_dock", "status", "bike_target", "dock_target"]
summary = []
for i, day in enumerate(days):
    frame = polls.filter(F.to_date("prediction_time") == F.lit(day)).toPandas()
    scored = lib.score_stations(frame, models, thresholds)[OUT]
    scored["bike_model_version"], scored["dock_model_version"] = versions["bike"], versions["dock"]
    summary.append(scored.assign(day=str(day)))
    if PUBLISH:
        (spark.createDataFrame(scored).write.mode("overwrite" if i == 0 else "append")
              .option("overwriteSchema", "true" if i == 0 else "false").saveAsTable(cfg.PLAYBACK))
    print(f"{day}: {len(scored):,} rows")

In [ ]:
# How did the warnings do? Of stations flagged at risk, how many actually ran out within 15 minutes?
allrows = pd.concat(summary)
check = []
for t, flag in (("bike", "empty_risk"), ("dock", "full_risk")):
    labeled = allrows[allrows[f"{t}_target"].notna()]
    warned = labeled[labeled.status == flag]
    check.append({"target": t, "labeled_rows": len(labeled), "warnings": len(warned),
                  "warning_hit_rate": warned[f"{t}_target"].mean(),
                  "base_rate": labeled[f"{t}_target"].mean(),
                  "events_caught": warned[f"{t}_target"].sum() / max(labeled[f"{t}_target"].sum(), 1)})
display(pd.DataFrame(check))
display(allrows.groupby(["day", "status"]).size().unstack(fill_value=0).reset_index())
if not PUBLISH:
    print("dev: table not written")